In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
import numpy as np

class EEGNet(nn.Module):
    def __init__(self, chans=22, samples=1001):
        super(EEGNet, self).__init__()

        self.chans = chans
        self.samples = samples

        # Layer 1: temporal/spatial setup for 22 channels
        self.conv1 = nn.Conv2d(1, 16, (1, chans), padding=0)
        self.batchnorm1 = nn.BatchNorm2d(16, affine=False)

        # Layer 2
        self.padding1 = nn.ZeroPad2d((16, 17, 0, 1))
        self.conv2 = nn.Conv2d(1, 4, (2, 32))
        self.batchnorm2 = nn.BatchNorm2d(4, affine=False)
        self.pooling2 = nn.MaxPool2d(2, 4)

        # Layer 3
        self.padding2 = nn.ZeroPad2d((2, 1, 4, 3))
        self.conv3 = nn.Conv2d(4, 4, (8, 4))
        self.batchnorm3 = nn.BatchNorm2d(4, affine=False)
        self.pooling3 = nn.MaxPool2d((2, 4))

        # Automatically calculate FC input size
        with torch.no_grad():
            dummy = torch.zeros(1, 1, samples, chans)
            dummy_out = self._features(dummy)
            self.flatten_dim = dummy_out.reshape(1, -1).shape[1]

        self.fc1 = nn.Linear(self.flatten_dim, 4)

    def _features(self, x):
        # Layer 1
        x = F.elu(self.conv1(x))
        x = self.batchnorm1(x)
        x = F.dropout(x, 0.25, training=self.training)
        x = x.permute(0, 3, 1, 2)

        # Layer 2
        x = self.padding1(x)
        x = F.elu(self.conv2(x))
        x = self.batchnorm2(x)
        x = F.dropout(x, 0.25, training=self.training)
        x = self.pooling2(x)

        # Layer 3
        x = self.padding2(x)
        x = F.elu(self.conv3(x))
        x = self.batchnorm3(x)
        x = F.dropout(x, 0.25, training=self.training)
        x = self.pooling3(x)

        return x

    def forward(self, x):
        x = self._features(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc1(x)
        return x

net = EEGNet().float().cuda(0)
print(net.forward(Variable(torch.Tensor(np.random.rand(1, 1, 1001, 22)).cuda(0))))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters())

tensor([[-1.4755,  0.5571,  0.6082,  0.6794]], device='cuda:0',
       grad_fn=<AddmmBackward0>)


#### Evaluate function 

In [3]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, recall_score, precision_score
import torch
import numpy as np
def evaluate(model, X, Y, params):
    model.eval()

    with torch.no_grad():
        inputs = torch.from_numpy(X).float().cuda(0)
        outputs = model(inputs)

        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        pred_labels = np.argmax(probs, axis=1)

    results = []

    for param in params:
        if param == "acc":
            results.append(accuracy_score(Y, pred_labels))

        elif param == "auc":
            try:
                results.append(
                    roc_auc_score(
                        Y,
                        probs,
                        multi_class="ovo",
                        labels=np.arange(probs.shape[1])
                    )
                )
            except ValueError:
                results.append(None)

        elif param == "fmeasure":
            results.append(f1_score(Y, pred_labels, average="macro"))

    model.train()
    return results

#### data

##### Data format:
X.shape - (#samples, 1, #timepoints,  #channels) <br>
Y.shape - (#samples)

In [4]:
X1=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X1.npy")
X1=X1.reshape(X1.shape[0], 1, X1.shape[2], X1.shape[1])

X2=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X2.npy")
X2=X2.reshape(X2.shape[0], 1, X2.shape[2], X2.shape[1])

X3=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X3.npy")
X3=X3.reshape(X3.shape[0], 1, X3.shape[2], X3.shape[1])

X4=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X4.npy")
X4=X4.reshape(X4.shape[0], 1, X4.shape[2], X4.shape[1])

X5=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X5.npy")
X5=X5.reshape(X5.shape[0], 1, X5.shape[2], X5.shape[1])

X6=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X6.npy")
X6=X6.reshape(X6.shape[0], 1, X6.shape[2], X6.shape[1])

X_train=np.append(X1, X2,0)
X_train=np.append(X_train, X3,0)
X_train=np.append(X_train, X4,0)
X_train=np.append(X_train, X5,0)
X_train=np.append(X_train, X6,0)

X_train.shape


(3456, 1, 1001, 22)

In [5]:
X1=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X1L.npy")

X2=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X2L.npy")

X3=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X3L.npy")

X4=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X4L.npy")

X5=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X5L.npy")

X6=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X6L.npy")

y_train=np.append(X1, X2,0)
y_train=np.append(y_train, X3,0)
y_train=np.append(y_train, X4,0)
y_train=np.append(y_train, X5,0)
y_train=np.append(y_train, X6,0)

y_train.shape

(3456,)

In [6]:
X7=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X7.npy")
X7=X7.reshape(X7.shape[0], 1, X7.shape[2], X7.shape[1])

X8=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X8.npy")
X8=X8.reshape(X8.shape[0], 1, X8.shape[2], X8.shape[1])

X_val=np.append(X7, X8,0)

X_val.shape

(1152, 1, 1001, 22)

In [7]:
X7=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X7L.npy")

X8=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X8L.npy")

y_val=np.append(X7, X8,0)

y_val.shape

(1152,)

In [8]:
X9=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X9.npy")
X9=X9.reshape(X9.shape[0], 1, X9.shape[2], X9.shape[1])
X_test=X9
X_test.shape

(576, 1, 1001, 22)

In [9]:
X9=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X9L.npy")
y_test=X9
y_test.shape

(576,)

In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_val_encoded = le.transform(y_val)
y_test_encoded = le.transform(y_test)

#### Run

In [11]:
batch_size = 32

for epoch in range(100):  # loop over the dataset multiple times
    print("\nEpoch ", epoch)
    
    running_loss = 0.0
    for i in range(len(X_train)//batch_size):
        s = i*batch_size
        e = i*batch_size+batch_size
        
        inputs = torch.from_numpy(X_train[s:e]).float()    
        inputs = inputs.cuda(0)
        labels = torch.tensor(
            y_train_encoded[s:e],
            dtype=torch.long,
            device="cuda"
        )
        
        # wrap them in Variable
        inputs, labels = Variable(inputs.cuda(0)), Variable(labels.cuda(0))

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        
        optimizer.step()
        
        running_loss += loss.item()
    
    # Validation accuracy
    params = ["acc", "auc", "fmeasure"]
    print(params)
    print("Training Loss ", running_loss)
    print("Train - ", evaluate(net, X_train, y_train_encoded, params))
    print("Validation - ", evaluate(net, X_val, y_val_encoded, params))
print("Test - ", evaluate(net, X_test, y_test_encoded, params))


Epoch  0
['acc', 'auc', 'fmeasure']
Training Loss  156.91071462631226
Train -  [0.3194444444444444, np.float64(0.5937937041859568), 0.2875927559312229]
Validation -  [0.2838541666666667, np.float64(0.5634157785172326), 0.25957700929341987]

Epoch  1
['acc', 'auc', 'fmeasure']
Training Loss  148.1642450094223
Train -  [0.3208912037037037, np.float64(0.6096823247099338), 0.26845369847131306]
Validation -  [0.2855902777777778, np.float64(0.5630576051311728), 0.24408207997766437]

Epoch  2
['acc', 'auc', 'fmeasure']
Training Loss  145.27020978927612
Train -  [0.33015046296296297, np.float64(0.6227642769240113), 0.26576061160936576]
Validation -  [0.2855902777777778, np.float64(0.5579055346579218), 0.23903498923792815]

Epoch  3
['acc', 'auc', 'fmeasure']
Training Loss  144.9315526485443
Train -  [0.33912037037037035, np.float64(0.6284303599751372), 0.2714914534171189]
Validation -  [0.2855902777777778, np.float64(0.5622960471322016), 0.24355741578858836]

Epoch  4
['acc', 'auc', 'fmeasure